# Text Classification — Zero-Shot with Groq

In [ ]:
import os
import time
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from groq import Groq

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

env_candidates = [
    Path(".env"),
    Path("shot") / ".env",
]
loaded = False
for env_path in env_candidates:
    if env_path.exists():
        if load_dotenv is not None:
            load_dotenv(env_path)
        else:
            for line in env_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip().strip('"').strip("'")
        print(f"Loaded .env from: {env_path.resolve()}")
        loaded = True
        break
if not loaded:
    print("No .env file found in expected paths (.env, shot/.env).")

# -- CONFIG -----------------------------------------------------------------
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
MODEL = "llama-3.3-70b-versatile"
SLEEP_SEC = 1.0
# ---------------------------------------------------------------------------

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found. Add it to .env in this folder (or shot/.env).")

client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready")

✅ Groq client ready


## 1. Load Data
Load the 3 submission CSVs (same structure: `ID;Text`). Put them all in the same folder.

In [ ]:
print("FILES:", FILES)
print("len(all_df):", len(all_df))
print("len(predictions):", len(predictions))
print("len(results_df):", len(results_df))
print("len(final_df):", len(final_df))
print(final_df["ID"].head(10).tolist())

In [ ]:

FILES = {

    "subm3": "subm3.csv",
}

dfs = {}
for name, path in FILES.items():
    df = pd.read_csv(path, sep=";", encoding="utf-8-sig")
    dfs[name] = df
    print(f"{name}: {len(df)} rows — columns: {list(df.columns)}")

# Combine all into one DataFrame for batch processing
all_df = pd.concat(dfs.values(), ignore_index=True)
print(f"\nTotal rows: {len(all_df)}")
all_df.head()

subm3: 150 rows — columns: ['ID', 'Text']

Total rows: 150


,ID,Text
0,D2-126,The reality about the places that diamonds are...
1,D2-127,Geothermobarometric calculations for a worldwi...
2,D2-128,Diamonds are formed deep within the Earth’s ma...
3,D2-129,Diamond is a solid form of the element carbon ...
4,D2-130,Diamonds are formed deep within the Earth unde...


## 2. Define Your Labels

Paste the full list of valid labels here. These come from your training set (the `~300 labels` you mentioned).  
The format should be a Python list of strings.

In [ ]:

LABELS = [
    "Human",
    "OpenAI",
    "Google",
    "Meta",
    "Anthropic",
]

LABELS = sorted({str(l).strip() for l in LABELS if str(l).strip()})
VALID_LABELS_LOWER = {l.lower(): l for l in LABELS}

print(f"Total labels defined: {len(LABELS)}")
print("First 5:", LABELS[:5])

Total labels defined: 5
First 5: ['Human', 'OpenAI', 'Google', 'Meta', 'Anthropic']


## 3. Zero-Shot Classification Function



In [ ]:
LABELS_STR = "\n".join(f"- {l}" for l in LABELS)

SYSTEM_PROMPT = """You are a precise text classification assistant.
Classify each text into exactly one label from the provided label list.
Use only labels from the list and output only the label name (no extra text)."""


def normalize_pred_label(pred: str) -> str:
    pred_clean = str(pred).strip().strip('"\'`').rstrip(".,;:").strip()
    return VALID_LABELS_LOWER.get(pred_clean.lower(), pred_clean)


def classify_zero_shot(text: str) -> str:
    """Classify a single text using zero-shot prompting."""
    text = "" if text is None else str(text)
    user_prompt = f"""Valid labels:

{LABELS_STR}

Text to classify:
\"\"\"
{text[:3000]}
\"\"\"

Return exactly one label from the valid labels list."""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=20,
    )
    raw = response.choices[0].message.content
    return normalize_pred_label(raw)


# Quick test
sample_text = all_df["Text"].iloc[0]
print("Sample text (first 200 chars):", sample_text[:200])
print("\nPredicted label:", classify_zero_shot(sample_text))

Sample text (first 200 chars): The reality about the places that diamonds are found is that our planet might host enormous quantities of diamonds below its rocky surface. However, it is only the diamonds that are found at the surfa

Predicted label: Human


## 4. Run Inference on All Rows

In [ ]:
predictions = []
errors      = []

for idx, row in tqdm(all_df.iterrows(), total=len(all_df), desc="Classifying"):
    try:
        label = classify_zero_shot(row["Text"])
        predictions.append({"ID": row["ID"], "Label": label})
    except Exception as e:
        print(f"Error on ID {row['ID']}: {e}")
        errors.append(row["ID"])
        predictions.append({"ID": row["ID"], "Label": "ERROR"})
    time.sleep(SLEEP_SEC)

print(f"\nDone. Errors: {len(errors)}")
if errors:
    print("Failed IDs:", errors)

Classifying: 100%|██████████| 150/150 [07:18<00:00,  2.92s/it]


✅ Done. Errors: 0


## 5. Post-process: Snap to Closest Valid Label

In [ ]:
from difflib import get_close_matches

results_df = pd.DataFrame(predictions)

def snap_label(pred: str, valid_labels: list, cutoff: float = 0.6) -> str:
    """Return the closest valid label, or the original if no good match."""
    pred = normalize_pred_label(pred)
    if pred in valid_labels:
        return pred
    matches = get_close_matches(pred, valid_labels, n=1, cutoff=cutoff)
    return matches[0] if matches else pred

results_df["Label_snapped"] = results_df["Label"].apply(
    lambda x: snap_label(x, LABELS)
)

# Show any corrections
corrections = results_df[results_df["Label"] != results_df["Label_snapped"]]
print(f"Corrections made: {len(corrections)}")
corrections[["ID", "Label", "Label_snapped"]].head(10)

Corrections made: 0


,ID,Label,Label_snapped


## 6. Save Submissions
One output file per dataset.

In [ ]:
final_df = results_df[["ID", "Label_snapped"]].rename(columns={"Label_snapped": "Label"})

out_path = "sub3-g5-MECD-A.csv"
final_df.to_csv(out_path, sep=";", index=False)
print(f"Saved {len(final_df)} rows -> {out_path}")

final_df.head(10)

Saved 150 rows -> submission_zeroshot_subm3.csv


,ID,Label
0,D2-126,Human
1,D2-127,Human
2,D2-128,Human
3,D2-129,Human
4,D2-130,Human
5,D2-131,Human
6,D2-132,Human
7,D2-133,Human
8,D2-134,Human
9,D2-135,Human
